In [1]:
import os
import pandas as pd

# --- ENVIRONMENT SELECTION ---
RUNNING_IN_KAGGLE = True # Keep False for Colab training!

if RUNNING_IN_KAGGLE:
    INPUT_PATH = "/kaggle/input/competitions/multilingual-speech-recognition"
    TRAIN_CSV = os.path.join(INPUT_PATH, "train.csv")
    TEST_CSV = os.path.join(INPUT_PATH, "test.csv")
    TRAIN_DIR = os.path.join(INPUT_PATH, "competition_data", "train")
    TEST_DIR = os.path.join(INPUT_PATH, "competition_data", "test")
    SUBMISSION_PATH = "/kaggle/working/submission.csv"
else:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')

    if not os.path.exists('/content/competition_data'):
        # Extracting to Colab's local storage for faster read speeds
        !unzip -q /content/drive/MyDrive/multilingual-speech-recognition.zip -d /content/competition_data
        print("✅ Dataset unzipped!")

    BASE_EXTRACT_DIR = "/content/competition_data"
    TRAIN_CSV = os.path.join(BASE_EXTRACT_DIR, "train.csv")
    TEST_CSV = os.path.join(BASE_EXTRACT_DIR, "test.csv")
    TRAIN_DIR = os.path.join(BASE_EXTRACT_DIR, "competition_data", "train")
    TEST_DIR = os.path.join(BASE_EXTRACT_DIR, "competition_data", "test")
    SUBMISSION_PATH = "/content/submission.csv"

print(f"📍 Mode: {'Kaggle' if RUNNING_IN_KAGGLE else 'Colab'}")

Mounted at /content/drive
✅ Dataset unzipped!
📍 Mode: Colab


In [ ]:
!pip install -q transformers datasets librosa evaluate jiwer accelerate

import torch
import librosa
from datasets import Dataset
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
import dataclasses
from typing import Any, Dict, List, Union

print("Loading Whisper Small...")
processor = WhisperProcessor.from_pretrained("openai/whisper-small", language="Hindi", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.generation_config.language = "<|en|>" # Set default for English retention

# 1. Prepare Training Dataset
train_df = pd.read_csv(TRAIN_CSV)
# Optional: If the dataset is huge and you are running out of time, uncomment the next line to train on a 50% subset
# train_df = train_df.sample(frac=0.5, random_state=42)

train_dataset = Dataset.from_pandas(train_df)

def prepare_dataset(batch):
    audio_path = os.path.join(TRAIN_DIR, batch["audio"])
    audio_array, _ = librosa.load(audio_path, sr=16000)
    batch["input_features"] = processor.feature_extractor(audio_array, sampling_rate=16000).input_features[0]
    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    return batch

print("Extracting features (This takes time)...")
train_dataset = train_dataset.map(prepare_dataset, remove_columns=train_dataset.column_names)

# 2. Data Collator
@dataclasses.dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

# 3. Training Setup (Optimized for Colab T4)
training_args = Seq2SeqTrainingArguments(
    output_dir="/content/whisper-small-finetuned",
    per_device_train_batch_size=4, # Keep this small for full fine-tuning!
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    num_train_epochs=3, # As advised by the 0.1 WER competitor
    gradient_checkpointing=True, # Critical to prevent crashes
    fp16=True,
    eval_strategy="no",
    save_strategy="epoch",
    logging_steps=25,
    remove_unused_columns=False,
    label_names=["labels"],
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    data_collator=data_collator,
    processing_class =processor.feature_extractor,
)

print("🚀 Starting 3-Epoch Full Fine-Tuning...")
trainer.train()

# Save locally to Colab
model.save_pretrained("/content/final_whisper_small")
processor.save_pretrained("/content/final_whisper_small")
print("✅ Training Complete!")

Loading Whisper Small...


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Extracting features (This takes time)...


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

🚀 Starting 3-Epoch Full Fine-Tuning...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
import torch
import librosa
import pandas as pd
from tqdm import tqdm
from transformers import pipeline, AutoModelForCTC, AutoProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading your fine-tuned Whisper model for English...")
ft_whisper_pipe = pipeline(
    "automatic-speech-recognition",
    model="/content/final_whisper_small", # Your trained model
    device=device,
)

print("Loading AI4Bharat Conformer for Indic Languages...")
indic_processor = AutoProcessor.from_pretrained("ai4bharat/indic-conformer-600m-multilingual")
indic_model = AutoModelForCTC.from_pretrained("ai4bharat/indic-conformer-600m-multilingual").to(device)

# Base Whisper purely for fast Language ID
lid_pipe = pipeline("automatic-speech-recognition", model="openai/whisper-tiny", device=device)

test_df = pd.read_csv(TEST_CSV)
results = []

print("🚀 Starting Hybrid Inference...")
for index, row in tqdm(test_df.iterrows(), total=len(test_df)):
    audio_path = os.path.join(TEST_DIR, row["audio"])
    audio, sr = librosa.load(audio_path, sr=16000)

    # 1. Detect Language
    lid_result = lid_pipe(audio, generate_kwargs={"task": "transcribe"})
    detected_lang = lid_result.get("chunks", [{}])[0].get("language", "en")

    # 2. Route the Audio
    if detected_lang == "en":
        # ENGLISH -> Use your Fine-Tuned Whisper
        result = ft_whisper_pipe(audio)
        transcription = result["text"].lower().strip() # Lowercase as advised

    else:
        # INDIC -> Use AI4Bharat Conformer (Zero-Shot)
        lang_map = {"ur": "hi", "ml": "ta"} # The "Bonus" advice
        target_lang = lang_map.get(detected_lang, detected_lang)

        # Conformer CTC Inference
        inputs = indic_processor(audio, sampling_rate=16000, return_tensors="pt").to(device)
        with torch.no_grad():
            logits = indic_model(**inputs).logits
        predicted_ids = torch.argmax(logits, dim=-1)
        transcription = indic_processor.batch_decode(predicted_ids)[0].strip()

    results.append({"audio": row["audio"], "text": transcription})

# Save Final Submission
submission_df = pd.DataFrame(results)
submission_df.to_csv(SUBMISSION_PATH, index=False, encoding='utf-8-sig')
print(f"🎉 Success! Download your submission from: {SUBMISSION_PATH}")